In [ ]:
# Operations Performed for Creating Model

# Importing Modules
# Loading Dataset
# EDA (Exploratory Data Analysis)
# Feature Extraction
# Model Creation
# Plot Model present in Tensorflow 
# Model Training
# Plot Result of Gender Graph
# Plot Result of Age Graph
# Model Prediction

## Importing Modules

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')
%matplotlib inline

import tensorflow as tf
from keras.preprocessing.image import load_img
from keras.models import Sequential, Model
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D, Input

## Loading Dataset

In [ ]:
# Access the location where the dataset is present (Loading the Dataset)
BASE_DIR = '/kaggle/input/fulldataset/Full Dataset/UTKFace/'

In [ ]:
# Age and Gender
def image_age_gender_paths(FILE_NAME):
    image_paths = []    # Store the path of every image
    age_labels = []     # Stores the age of every image
    gender_labels = []  # Stores the gender of every image (0=Male, 1=Female)

    # Traversing through all the images

    # tqdm is a iterator which will iterate over every image one by one
#     for filename in tqdm(os.listdir(BASE_DIR)):

    for filename in tqdm(os.listdir(FILE_NAME)):


        # image_path will store the loaction of each image with image name
#       image_path = os.path.join(BASE_DIR, filename) 
        image_path = os.path.join(FILE_NAME, filename) 

        # temp will store the image name after splitting it from '_'
        temp = filename.split('_')

        # age will store the age of each image 
        age = int(temp[0])

        # gender will store the age of each image 
        gender = int(temp[1])
    #     if(gender==0):
    #         gender="Male"
    #     else:
    #         gender="Female"

        # image_paths will store all the results stored in image_path
        image_paths.append(image_path)

        # age_labels will store all the results stored in age
        age_labels.append(age)
    
        # gender_labels will store all the results stored in gender
        gender_labels.append(gender)
        
    return image_paths,age_labels,gender_labels

In [ ]:
image_paths = []    # Store the path of every image
age_labels = []     # Stores the age of every image
gender_labels = []  # Stores the gender of every image (0=Male, 1=Female)
image_paths,age_labels,gender_labels = image_age_gender_paths(BASE_DIR)

In [ ]:
# Converting to dataframe

def convert_to_dataframe(image_paths,age_labels,gender_labels):
    # Dataframe() Creates a 2dimensional or 3dimensional array, or a table with rows and columns.
    df = pd.DataFrame() 

    # df['saba'],df['saif'],df['shiraz']=[[1,2,3],[4,5,6],[7,8,9]]
    #     saba     saif   shiraz
    # 0    1        4       7
    # 1    2        5       8
    # 2    3        6       9

    df['image'], df['age'], df['gender'] = image_paths, age_labels, gender_labels
    df.head()
    return df

In [ ]:
  # Dataframe() Creates a 2dimensional or 3dimensional array, or a table with rows and columns.
# df = pd.DataFrame()
df = convert_to_dataframe(image_paths,age_labels,gender_labels)

In [ ]:
df.head(10)

In [ ]:
# Label 0 as Male and 1 as Female
gender_label = {0:'Male', 1:'Female'}

## Exploratory Data Analysis

In [ ]:
# PIL stands for Python Imaging Library, 
# and it's the original library that enabled Python to deal with images. 

from PIL import Image
img = Image.open(df['image'][10])  # store image in img 
plt.axis('off')
plt.imshow(img); # Display the image stored in img

In [ ]:
# Distplot is a distribution plot.
# It distributes the elements and display it in the fomm of graph. 

sns.distplot(df['age'])

In [ ]:
# Distplot is a distribution plot.
# It distributes the elements and display it in the fomm of graph. 

sns.distplot(df['gender'])

In [ ]:
# Countplot count the number of element.
# It count the elements and display it in the form of graph of male and female. 

sns.countplot(df['gender'])

In [ ]:
# To Display Image in form of Grid of Images

plt.figure(figsize=(20, 20)) # Plot the image in size 20 X 20
# .iloc[] is primarily integer position based (from 0 to length-1 of the axis)
images = df.iloc[0:25]        # Extract the first 25 index of df and store in files

# .itertuples Iterate over DataFrame rows as namedtuples.
for index, image, age, gender in images.itertuples():
    
    # Plot the image in 5 X 5 grid
    plt.subplot(5, 5, index+1)
    
    # Load image in img
    img = load_img(image)
#     print(img)

    # Convert the img in array form
    img = np.array(img)
#     print(img)
    
    # Plot the image
    plt.imshow(img)
    
    # Plot the gender
    plt.title(f"Age: {age} Gender: {gender_label[gender]}")
    
    plt.axis('off')

## Feature Extraction

In [ ]:
def feature_extraction(images):
    features = []
    
    for image in tqdm(images):
        img = load_img(image,grayscale=True)
#         print(img)
#         plt.imshow(img)

#          Aliasing is the visual stair-stepping of edges that occurs in an image 
#         when the resolution is too low. Anti-aliasing is the smoothing of jagged edges 
#         in digital images by averaging the colors of the pixels at a boundary.
        img = img.resize((128,128),Image.ANTIALIAS)
    
#         print(img)
#         plt.imshow(img)

        img = np.array(img)
#         print(img)

        features.append(img)
#         print(features)

    features = np.array(features)
#     print(features)

#     ignore this step if using RGB
    features = features.reshape(len(features), 128, 128, 1)
    
    return features

In [ ]:
# Stores all the feature extraction in feature_extracted
feature_extracted = feature_extraction(df['image'])

In [ ]:
# print the shape of feature_extracted

feature_extracted.shape

In [ ]:
# Normalize the image

feature_extracted = feature_extracted/255.0

In [ ]:
# store_gender stores the gender of all the images.
# store_age stores the age of all the images.

store_gender = np.array(df['gender'])
store_age = np.array(df['age'])

In [ ]:
input_shape = (128,128,1)

## Model Creation

## Convolutional Layers

In [ ]:
inputs = Input((input_shape))

convolutional_layer_1 = Conv2D(32,kernel_size=(3,3),activation='relu')(inputs)
max_pooling_1 = MaxPooling2D(pool_size=(2,2)) (convolutional_layer_1)
convolutional_layer_2 = Conv2D(64,kernel_size=(3,3), activation='relu') (max_pooling_1)
max_pooling_2 = MaxPooling2D(pool_size=(2,2)) (convolutional_layer_2)
convolutional_layer_3 = Conv2D(64,kernel_size=(3,3), activation='relu') (max_pooling_2)
max_pooling_3 = MaxPooling2D(pool_size=(2,2)) (convolutional_layer_3)
convolutional_layer_4 = Conv2D(64,kernel_size=(3,3), activation='relu') (max_pooling_3)
max_pooling_4 = MaxPooling2D(pool_size=(2,2)) (convolutional_layer_4)

flatten = Flatten() (max_pooling_4)


# Fully Connected Layers

dense_1 = Dense(256, activation='relu') (flatten)
dense_2 = Dense(256, activation='relu') (flatten)

dropout_1 = Dropout(0.3) (dense_1)
dropout_2 = Dropout(0.3) (dense_2)

output_1 = Dense(1, activation='sigmoid', name='gender_out') (dropout_1)
output_2 = Dense(1, activation='relu', name='age_out') (dropout_2)

model = Model(inputs=[inputs], outputs=[output_1, output_2])

model.compile(loss=['binary_crossentropy', 'mae'], optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# Plot the model

from tensorflow.keras.utils import plot_model
plot_model(model)

In [ ]:
# Train Model

# train_model = model.fit(x=feature_extracted, y=[store_gender, store_age], batch_size=32, epochs=30, validation_split=0.2)
train_model = model.fit(x=feature_extracted, y=[store_gender, store_age], batch_size=32, epochs=30, validation_split=0.2)

In [ ]:
# Plot results for Gender

training_accuracy = train_model.history['gender_out_accuracy']
validation_accuracy = train_model.history['val_gender_out_accuracy']
epochs = range(len(training_accuracy))

plt.plot(epochs,training_accuracy,'b',label='Training Accuracy')
plt.plot(epochs,validation_accuracy,'r',label='Validation Accuracy')
plt.title('Accuracy Graph')
plt.legend()
plt.figure()

training_accuracy_loss = train_model.history['gender_out_loss']
validation_loss = train_model.history['val_gender_out_loss']

plt.plot(epochs,training_accuracy_loss,'b',label='Training Accuracy Loss')
plt.plot(epochs,validation_loss,'r',label='Validation Accuracy Loss')
plt.title('Loss Graph')
plt.legend()
plt.show()


In [ ]:
# Plot results for Age


training_accuracy = train_model.history['age_out_accuracy']
validation_accuracy = train_model.history['val_age_out_accuracy']
epochs = range(len(training_accuracy))

plt.plot(epochs,training_accuracy,'b',label='Training Accuracy')
plt.plot(epochs,validation_accuracy,'r',label='Validation Accuracy')
plt.title('Accuracy Graph')
plt.legend()
plt.figure()

training_accuracy_loss = train_model.history['age_out_loss']
validation_loss = train_model.history['val_age_out_loss']

plt.plot(epochs,training_accuracy_loss,'b',label='Training Accuracy Loss')
plt.plot(epochs,validation_loss,'r',label='Validation Accuracy Loss')
plt.title('Loss Graph')
plt.legend()
plt.show()


## Prediction with Test Data

In [ ]:
image_index = 150
print("Original Gender = ",gender_label[store_gender[image_index]], "Original Age = ",store_age[image_index])

# Predict from model

predict_from_model = model.predict(feature_extracted[image_index].reshape(1,128,128,1))
predict_gender = gender_label[round(predict_from_model[0][0][0])]
predict_age = round(predict_from_model[1][0][0])

print("Predicted Gender = ",predict_gender,"Predicted Age = ",predict_age)
plt.axis('off')
plt.imshow(feature_extracted[image_index].reshape(128,128),cmap='gray')

In [ ]:
image_index = 222
print("Original Gender = ",gender_label[store_gender[image_index]], "Original Age = ",store_age[image_index])

# Predict from model

predict_from_model = model.predict(feature_extracted[image_index].reshape(1,128,128,1))
predict_gender = gender_label[round(predict_from_model[0][0][0])]
predict_age = round(predict_from_model[1][0][0])

print("Predicted Gender = ",predict_gender,"Predicted Age = ",predict_age)
plt.axis('off')
plt.imshow(feature_extracted[image_index].reshape(128,128),cmap='gray')

In [ ]:
image_indexs = [222,1,2,3,4,5,6,7,8,9,10]
for image_index in image_indexs:
    print("Original Gender = ",gender_label[store_gender[image_index]], "Original Age = ",store_age[image_index])

    # Predict from model

    predict_from_model = model.predict(feature_extracted[image_index].reshape(1,128,128,1))
    predict_gender = gender_label[round(predict_from_model[0][0][0])]
    predict_age = round(predict_from_model[1][0][0])

    print("Predicted Gender = ",predict_gender,"Predicted Age = ",predict_age)
    plt.axis('off')
    plt.imshow(feature_extracted[image_index].reshape(128,128),cmap='gray')
    plt.show()

## Prediction with External Folder

In [ ]:
TEST_DIR = '/kaggle/input/fulldataset/Full Dataset/TestImage/'

In [ ]:
test_image_paths = []
test_age_labels = []
test_gender_labels = []

test_image_paths, test_age_labels, test_gender_labels=image_age_gender_paths(TEST_DIR)

test_df = convert_to_dataframe(test_image_paths, test_age_labels, test_gender_labels)          

In [ ]:
feature_extracted2 = feature_extraction(test_df['image'])
feature_extracted2 = feature_extracted2/255.0 

In [ ]:
# store_gender stores the gender of all the images.
# store_age stores the age of all the images.

test_store_gender = np.array(test_df['gender'])
test_store_age = np.array(test_df['age'])

In [ ]:
image_index = 111
print("Original Gender = ",gender_label[test_store_gender[image_index]], "Original Age = ",test_store_age[image_index])

# Predict from model

predict_from_model = model.predict(feature_extracted2[image_index].reshape(1,128,128,1))
predict_gender = gender_label[round(predict_from_model[0][0][0])]
predict_age = round(predict_from_model[1][0][0])

print("Predicted Gender = ",predict_gender,"Predicted Age = ",predict_age)
plt.axis('off')
plt.imshow(feature_extracted2[image_index].reshape(128,128),cmap='gray')

In [ ]:
images=test_df.iloc[:20]
for image_index, image, age, gender in images.itertuples():
    print(image_index)
    print("Original Gender = ",gender_label[test_store_gender[image_index]], "Original Age = ",test_store_age[image_index])

    # Predict from model

    predict_from_model = model.predict(feature_extracted2[image_index].reshape(1,128,128,1))
    predict_gender = gender_label[round(predict_from_model[0][0][0])]
    predict_age = round(predict_from_model[1][0][0])

    print("Predicted Gender = ",predict_gender,"Predicted Age = ",predict_age)
    plt.axis('off')
    plt.imshow(feature_extracted2[image_index].reshape(128,128),cmap='gray')
    plt.show()

In [ ]:
val_gender_accuracy, val_age_accuracy = train_model.history['val_gender_out_accuracy'][-1], train_model.history['val_age_out_accuracy'][-1]
print('Validation Gender Accuracy:', val_gender_accuracy)
print('Validation Age Accuracy:', val_age_accuracy)

In [ ]:
val_gender_accuracy, val_age_accuracy = train_model.history['gender_out_accuracy'][-1], train_model.history['age_out_accuracy'][-1]
print('Gender Accuracy:', val_gender_accuracy)
print('Age Accuracy:', val_age_accuracy)